In [ ]:
import sys
!{sys.executable} -m pip install python-dotenv

In [1]:
"""
Test Job.get_next_request() integration with LLM_API.

This script verifies that the chat_history format returned by Job.get_next_request()
is compatible with LLM_API.process_request() method. It can be converted to a 
Jupyter notebook for interactive testing.
"""

import dbzero as db0
from collections import namedtuple
from statek.agent import Agent
from statek.executors.job import Job, JobDef, JobStatus
from statek.executors.utils import run_job_step
from statek.pyenv import PyEnv

In [2]:
# Initialize dbzero
db0.init(".dbzero_data")
db0.open("test-prefix-2")

In [3]:
# sample tools mocs

def add(a: int, b: int) -> int:
    """Adds two elements"""
    return a + b

def multiply(a: int, b: int) -> int:
    """multiply two elements"""
    return a * b

def exit(reason: str):
    print(reason)

In [5]:

"""Create a Job instance"""
# Create agent and pyenv
agent = Agent(_system_prompt="""You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: \n{tools} \nReturn only one call at time e. g sum(2,5). Don't calculate without tools provided in this prompt.
Send only operation wrapped in print().
No comments just for eg: print(add(1,2)). When finished call: exit('success'). """, _tools=[add,multiply], role="test")
pyenv = PyEnv(local_state={
    "multiply":multiply,
    "add":add
})


# Create job definition and job
job_def = JobDef(
    agent=agent,
    description="Solve this problem: {goal}",
    goal="3 * 5 + 4 * 2",
    warmup_code=None
)
job = Job(
    job_def=job_def,
    model_family="test",
    model="openai/gpt-5",
    job_status=JobStatus.READY,
    py_env=pyenv
)


### Test 1: Check if run_job_step can go through happy path

In [6]:
from dotenv import load_dotenv
load_dotenv("./.env")


True

In [7]:
result = await run_job_step(job)

LLM Response:
print(multiply(3,5))
----------------------------------------


In [8]:
print(result)

False


In [9]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: 
System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
Return only one call at time e. g sum(2,5). Don't calculate without tools provided in this prompt.
Send only operation wrapped in print().
No comments just for eg: print(add(1,2)). When finished call: exit('success'). 
History: 
Solve this problem: 3 * 5 + 4 * 2
print(multiply(3,5))


In [10]:
# run next iteration:

In [11]:
result = await run_job_step(job)

LLM Response:
print(multiply(4,2))
----------------------------------------


In [12]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: 
System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
Return only one call at time e. g sum(2,5). Don't calculate without tools provided in this prompt.
Send only operation wrapped in print().
No comments just for eg: print(add(1,2)). When finished call: exit('success'). 
History: 
Solve this problem: 3 * 5 + 4 * 2
print(multiply(3,5))
> 15

print(multiply(4,2))


In [13]:
# run 3 step
result = await run_job_step(job)

LLM Response:
print(add(15,8))
----------------------------------------


In [14]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: 
System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
Return only one call at time e. g sum(2,5). Don't calculate without tools provided in this prompt.
Send only operation wrapped in print().
No comments just for eg: print(add(1,2)). When finished call: exit('success'). 
History: 
Solve this problem: 3 * 5 + 4 * 2
print(multiply(3,5))
> 15

print(multiply(4,2))
> 8

print(add(15,8))


In [15]:
# run 4 step
result = await run_job_step(job)

LLM Response:
exit('success')
----------------------------------------


In [16]:
print(result)

False


In [17]:
# Get the next request parameters
request = job.get_next_request()

print(f"\nRequest prompt: {request['prompt']}")
print(f"System prompt: {request['system_prompt']}")
print("History: ")
for history in request['chat_history']:
    print(history)


Request prompt: 
System prompt: You are a helpful assistant. You solves given equations, 
but can only calculate one thin at a time. Need to use provided tools: 
>def add(a: int, b: int) -> int
>def multiply(a: int, b: int) -> int 
Return only one call at time e. g sum(2,5). Don't calculate without tools provided in this prompt.
Send only operation wrapped in print().
No comments just for eg: print(add(1,2)). When finished call: exit('success'). 
History: 
Solve this problem: 3 * 5 + 4 * 2
print(multiply(3,5))
> 15

print(multiply(4,2))
> 8

print(add(15,8))
> 23

exit('success')


In [18]:
# run final step
result = await run_job_step(job)
print(result)

True
